# Imports

In [19]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    recall_score, 
    precision_score, 
    f1_score, 
    roc_auc_score
)
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Data Ingestion & Encoding

In [20]:
df = pd.read_csv('data.csv')

# Drop unique identifiers and unmapped system artifacts
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

# Target encoding: Malignant -> 1, Benign -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print("Target Class Distribution:")
print(df['diagnosis'].value_counts())
print(f"\nDataset Shape: {df.shape}")

Target Class Distribution:
diagnosis
0    357
1    212
Name: count, dtype: int64

Dataset Shape: (569, 31)


# Data Splits, Scaling & Resampling

In [21]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Stratify to guarantee identical class distributions across splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling must precede SMOTE to preserve distance metric validity
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle minority class imbalance on training subset
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Pre-SMOTE Train Target Count:  {np.bincount(y_train)}")
print(f"Post-SMOTE Train Target Count: {np.bincount(y_train_resampled)}")

Pre-SMOTE Train Target Count:  [285 170]
Post-SMOTE Train Target Count: [285 285]


# Support Vector Machine (SVM) Evaluation

In [22]:
svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm.fit(X_train_resampled, y_train_resampled)

svm_preds = svm.predict(X_test_scaled)
svm_probs = svm.predict_proba(X_test_scaled)[:, 1]

svm_metrics = {
    'Accuracy': accuracy_score(y_test, svm_preds),
    'Recall': recall_score(y_test, svm_preds),
    'Precision': precision_score(y_test, svm_preds),
    'F1-Score': f1_score(y_test, svm_preds),
    'ROC-AUC': roc_auc_score(y_test, svm_probs)
}


# XGBoost Classifier Evaluation

In [23]:
xgb = XGBClassifier(
    n_estimators=100, 
    max_depth=3, 
    learning_rate=0.1, 
    random_state=42, 
    eval_metric='logloss'
)
xgb.fit(X_train_resampled, y_train_resampled)

xgb_preds = xgb.predict(X_test_scaled)
xgb_probs = xgb.predict_proba(X_test_scaled)[:, 1]

xgb_metrics = {
    'Accuracy': accuracy_score(y_test, xgb_preds),
    'Recall': recall_score(y_test, xgb_preds),
    'Precision': precision_score(y_test, xgb_preds),
    'F1-Score': f1_score(y_test, xgb_preds),
    'ROC-AUC': roc_auc_score(y_test, xgb_probs)
}



# Artificial Neural Network (ANN) Evaluation

In [24]:
# Fixed input layer syntax to comply with modern Keras standards
ann = Sequential([
    Input(shape=(X_train_resampled.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') 
])

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mitigate validation leakage by holding out a distinct fraction of resampled data
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

ann.fit(
    X_train_resampled, y_train_resampled,
    validation_split=0.1,  
    epochs=80,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0  # Prevents verbose epoch logs from cluttering execution files
)

ann_probs = ann.predict(X_test_scaled, verbose=0).flatten()
ann_preds = (ann_probs >= 0.5).astype(int)

ann_metrics = {
    'Accuracy': accuracy_score(y_test, ann_preds),
    'Recall': recall_score(y_test, ann_preds),
    'Precision': precision_score(y_test, ann_preds),
    'F1-Score': f1_score(y_test, ann_preds),
    'ROC-AUC': roc_auc_score(y_test, ann_probs)
}


# Performance Merging

In [25]:
comparison_df = pd.DataFrame({
    'SVM': svm_metrics,
    'XGBoost': xgb_metrics,
    'ANN': ann_metrics
}).T

# Production Pipeline


In [26]:
print("\n=== FINAL MODEL PERFORMANCE COMPARISON ===")
print(comparison_df.to_string())

# Medical context prioritizes ROC-AUC to balance classification thresholding risks
best_model_name = comparison_df['ROC-AUC'].idxmax()
print(f"\nSelected Model Variant for Deployment: {best_model_name}")

# Defend against missing directories during CI/CD execution runs
export_dir = '../backend/models'
os.makedirs(export_dir, exist_ok=True)
model_export_path = os.path.join(export_dir, 'best_model.joblib')

if best_model_name in ['SVM', 'XGBoost']:
    from sklearn.pipeline import Pipeline
    
    active_classifier = svm if best_model_name == 'SVM' else xgb
    prod_pipeline = Pipeline([
        ('scaler', scaler), 
        ('classifier', active_classifier)
    ])
    
    joblib.dump(prod_pipeline, model_export_path)
    print(f"Successfully serialized unified {best_model_name} Pipeline to: {model_export_path}")

else:
    # Separate weight matrices and scaling configuration objects for Keras deployment paths
    ann.save(os.path.join(export_dir, 'ann_weights.h5'))
    joblib.dump(scaler, os.path.join(export_dir, 'prod_scaler.joblib'))
    print(f"Successfully split and serialized ANN network configs to: {export_dir}/")


=== FINAL MODEL PERFORMANCE COMPARISON ===
         Accuracy    Recall  Precision  F1-Score   ROC-AUC
SVM      0.973684  0.952381    0.97561  0.963855  0.995701
XGBoost  0.964912  0.904762    1.00000  0.950000  0.996032
ANN      0.973684  0.928571    1.00000  0.962963  0.993386

Selected Model Variant for Deployment: XGBoost
Successfully serialized unified XGBoost Pipeline to: ../backend/models/best_model.joblib
